# predict DD

In [1]:
# import merged csv

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv("wide_diff_all_data.csv")
df


,subjectID,age_group,strata,Gender,DDisc_AUC_40K,Release,Acquisition,Age,3T_Full_MR_Compl,7T_Full_MR_Compl,...,Occipital_dki_mk,Orbital_dki_mk,PostParietal_dki_mk,SLF_L_dki_mk,SLF_R_dki_mk,SupFrontal_dki_mk,SupParietal_dki_mk,Temporal_dki_mk,UNC_L_dki_mk,UNC_R_dki_mk
0,100206,26-30,26-30_M,M,0.050000,S900,Q11,26-30,True,False,...,1.101557,1.084557,1.053842,1.107496,1.110725,1.112234,1.026142,1.056521,0.995955,0.988601
1,100307,26-30,26-30_F,F,0.311459,Q1,Q01,26-30,True,False,...,1.006925,1.045562,0.987310,1.008259,1.024426,0.984244,0.928578,0.976830,0.895127,0.915798
2,100610,26-30,26-30_M,M,0.868750,S900,Q08,26-30,True,True,...,0.996652,1.105037,1.005991,1.009783,1.057179,1.042984,0.986284,1.009254,0.956977,0.977282
3,101006,31-35,31-35_F,F,0.783073,S500,Q06,31-35,True,False,...,0.931849,1.074894,0.951102,1.011284,1.040509,1.017018,0.981339,0.962131,0.965962,1.004965
4,101107,22-25,22-25_M,M,0.584375,S500,Q06,22-25,True,False,...,0.971370,1.107099,0.975676,0.999982,1.025620,1.027364,0.963823,0.960659,0.917482,0.953508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
686,991267,26-30,26-30_M,M,0.338151,S500,Q06,26-30,True,False,...,1.013882,1.120605,1.032284,1.050530,1.096211,1.068493,1.012103,1.027903,0.920387,0.938566
687,992774,31-35,31-35_M,M,0.019531,Q2,Q02,31-35,True,False,...,0.988976,1.058804,1.013462,1.038346,1.066970,1.026199,0.991104,1.005351,0.923729,0.954652
688,993675,26-30,26-30_F,F,0.938281,S900,Q09,26-30,True,False,...,0.900823,0.980285,0.990303,1.035805,1.057302,1.012190,0.997047,0.953649,0.877470,0.861817
689,994273,26-30,26-30_M,M,0.529427,S500,Q06,26-30,True,False,...,1.005014,1.065678,0.950951,0.980729,1.005018,0.985756,0.905019,0.909975,0.917472,0.946772


In [3]:
unique_age_strings = sorted(df["age_group"].dropna().astype(str).unique())
print(unique_age_strings)

['22-25', '26-30', '31-35', '36+']


In [4]:
count_36_plus = (df["age_group"] == "36+").sum()
print(count_36_plus)

7


In [5]:
age_map = {
    "22-25": 23.5,
    "26-30": 28,
    "31-35": 33,
    "36+": 38
}

df["age_average"] = df["age_group"].map(age_map)

In [6]:
df[["age_group", "age_average"]].head(50)

,age_group,age_average
0,26-30,28.0
1,26-30,28.0
2,26-30,28.0
3,31-35,33.0
4,22-25,23.5
5,26-30,28.0
6,31-35,33.0
7,26-30,28.0
8,22-25,23.5
9,26-30,28.0


In [7]:
df["age_average"] = pd.to_numeric(df["age_average"], errors="coerce")


In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

X = df[diffusion_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.410275,0.410275
1,PC2,0.103745,0.514021
2,PC3,0.069292,0.583313
3,PC4,0.052875,0.636187
4,PC5,0.032736,0.668923
5,PC6,0.027673,0.696596
6,PC7,0.024380,0.720976
7,PC8,0.021719,0.742695
8,PC9,0.018320,0.761014
9,PC10,0.015834,0.776848


In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Diffusion-only columns
diffusion_cols = [col for col in df.columns if "dki_" in col]

X = df[diffusion_cols].copy()

# Make sure everything is numeric
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

# Standardize
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.410275,0.410275
1,PC2,0.103745,0.514021
2,PC3,0.069292,0.583313
3,PC4,0.052875,0.636187
4,PC5,0.032736,0.668923
5,PC6,0.027673,0.696596
6,PC7,0.024380,0.720976
7,PC8,0.021719,0.742695
8,PC9,0.018320,0.761014
9,PC10,0.015834,0.776848


In [10]:
# Add age and sex back in as covariates
pca_cov_df = pd.concat(
    [
        pca_df,
        df[["age_group", "Gender"]].copy()
    ],
    axis=1
)

pca_cov_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,age_group,Gender
0,6.422050,2.728150,4.368038,3.012170,1.579520,1.709322,-1.707827,0.706479,-1.421869,0.242638,...,0.548093,0.250652,0.773261,1.846650,2.005287,-1.808088,-0.844851,0.309854,26-30,M
1,-6.229065,1.435725,-3.672986,1.128096,1.371798,2.175690,-0.854667,1.366902,2.062569,1.246272,...,0.498026,-0.876906,0.678483,0.288881,2.073943,-0.281320,-0.379334,-1.085239,26-30,F
2,-8.605275,0.281892,6.608669,-0.139258,0.071598,0.007280,-0.498974,1.792533,1.244373,0.687242,...,-0.238380,0.639044,-0.548295,-0.036192,-0.019633,0.059623,-0.451544,0.530692,26-30,M
3,-7.330715,-0.609172,2.873511,-4.572229,-2.861640,-1.391004,1.682103,1.387887,-0.364662,-1.149462,...,0.737607,-1.135512,-0.161130,-0.276398,-1.236917,0.154336,0.353786,0.642891,31-35,F
4,-3.368465,2.158385,-1.731678,-2.789795,-0.329603,1.705612,-0.449386,-1.862459,-2.202574,0.296327,...,-0.851057,-0.586275,0.528154,-0.486835,0.268906,0.338131,0.098204,-0.485538,22-25,M


In [11]:
# Add age and sex back in as covariates
pca_cov_df = pd.concat(
    [
        pca_df,
        df[["age_average", "Gender"]].copy()
    ],
    axis=1
)

pca_cov_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,age_average,Gender
0,6.422050,2.728150,4.368038,3.012170,1.579520,1.709322,-1.707827,0.706479,-1.421869,0.242638,...,0.548093,0.250652,0.773261,1.846650,2.005287,-1.808088,-0.844851,0.309854,28.0,M
1,-6.229065,1.435725,-3.672986,1.128096,1.371798,2.175690,-0.854667,1.366902,2.062569,1.246272,...,0.498026,-0.876906,0.678483,0.288881,2.073943,-0.281320,-0.379334,-1.085239,28.0,F
2,-8.605275,0.281892,6.608669,-0.139258,0.071598,0.007280,-0.498974,1.792533,1.244373,0.687242,...,-0.238380,0.639044,-0.548295,-0.036192,-0.019633,0.059623,-0.451544,0.530692,28.0,M
3,-7.330715,-0.609172,2.873511,-4.572229,-2.861640,-1.391004,1.682103,1.387887,-0.364662,-1.149462,...,0.737607,-1.135512,-0.161130,-0.276398,-1.236917,0.154336,0.353786,0.642891,33.0,F
4,-3.368465,2.158385,-1.731678,-2.789795,-0.329603,1.705612,-0.449386,-1.862459,-2.202574,0.296327,...,-0.851057,-0.586275,0.528154,-0.486835,0.268906,0.338131,0.098204,-0.485538,23.5,M


In [12]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm

diffusion_cols = [col for col in df.columns if "dki_" in col]

X_diff = df[diffusion_cols].copy()
X_diff = X_diff.apply(pd.to_numeric, errors="coerce").fillna(X_diff.mean())

X_scaled = StandardScaler().fit_transform(X_diff)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

cov_df = df[["DDisc_AUC_40K", "age_average", "Gender"]].copy()
cov_df["DDisc_AUC_40K"] = pd.to_numeric(cov_df["DDisc_AUC_40K"], errors="coerce")
cov_df["age_average"] = pd.to_numeric(cov_df["age_average"], errors="coerce")

cov_df = pd.get_dummies(cov_df, columns=["Gender"], drop_first=True)

model_df = pd.concat([cov_df, pca_df], axis=1).dropna()

y = model_df["DDisc_AUC_40K"]
X = model_df.drop(columns=["DDisc_AUC_40K"])
X = sm.add_constant(X)

y = y.astype(float)
X = X.astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     1.962
Date:                Thu, 23 Jul 2026   Prob (F-statistic):            0.00272
Time:                        20:06:22   Log-Likelihood:                -82.960
No. Observations:                 691   AIC:                             221.9
Df Residuals:                     663   BIC:                             349.0
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.4133      0.097      4.268      

In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

X = df[diffusion_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.410275,0.410275
1,PC2,0.103745,0.514021
2,PC3,0.069292,0.583313
3,PC4,0.052875,0.636187
4,PC5,0.032736,0.668923
5,PC6,0.027673,0.696596
6,PC7,0.024380,0.720976
7,PC8,0.021719,0.742695
8,PC9,0.018320,0.761014
9,PC10,0.015834,0.776848


In [14]:
import pandas as pd
import statsmodels.api as sm

pc_cols = [col for col in pca_df.columns if col.startswith("PC")]

reg_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_average", "Gender"]],
        pca_df[pc_cols]
    ],
    axis=1
)

reg_df["DDisc_AUC_40K"] = pd.to_numeric(reg_df["DDisc_AUC_40K"], errors="coerce")
reg_df = pd.get_dummies(reg_df, columns=["age_average", "Gender"], drop_first=True)
reg_df = reg_df.dropna()

y = reg_df["DDisc_AUC_40K"].astype(float)
X = reg_df.drop(columns=["DDisc_AUC_40K"]).astype(float)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.076
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     1.867
Date:                Thu, 23 Jul 2026   Prob (F-statistic):            0.00412
Time:                        20:06:23   Log-Likelihood:                -82.319
No. Observations:                 691   AIC:                             224.6
Df Residuals:                     661   BIC:                             360.8
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.4715      0.029  

In [15]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1) Keep only diffusion columns with the substrings you specified
pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

print("Number of diffusion columns:", len(diffusion_cols))
print("First 10 diffusion columns:", diffusion_cols[:10])

# 2) Build PCA only from those columns
X_diff = df[diffusion_cols].copy()
X_diff = X_diff.apply(pd.to_numeric, errors="coerce")
X_diff = X_diff.fillna(X_diff.mean())

X_scaled = StandardScaler().fit_transform(X_diff)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

# 3) Confirm loadings only involve diffusion variables
loadings = pd.DataFrame(
    pca.components_.T,
    index=diffusion_cols,
    columns=[f"PC{i+1}" for i in range(25)]
)

print("\nTop loadings for PC19:")
display(
    loadings["PC19"]
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("abs_loading")
    .join(loadings["PC19"].to_frame("loading"))
)

# 4) Regression: DD score on PCs + age + sex
reg_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df
    ],
    axis=1
)

reg_df["DDisc_AUC_40K"] = pd.to_numeric(reg_df["DDisc_AUC_40K"], errors="coerce")
reg_df = pd.get_dummies(reg_df, columns=["age_group", "Gender"], drop_first=True)
reg_df = reg_df.dropna()

y = reg_df["DDisc_AUC_40K"].astype(float)
X = reg_df.drop(columns=["DDisc_AUC_40K"]).astype(float)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

Number of diffusion columns: 96
First 10 diffusion columns: ['ARC_L_dki_awf', 'ARC_R_dki_awf', 'ATR_L_dki_awf', 'ATR_R_dki_awf', 'AntFrontal_dki_awf', 'CGC_L_dki_awf', 'CGC_R_dki_awf', 'CST_L_dki_awf', 'CST_R_dki_awf', 'IFO_L_dki_awf']

Top loadings for PC19:


,abs_loading,loading
CST_R_dki_mk,0.316035,0.316035
PostParietal_dki_fa,0.288661,0.288661
ILF_R_dki_fa,0.268618,0.268618
CST_L_dki_mk,0.264021,0.264021
CST_L_dki_fa,0.223134,-0.223134
IFO_L_dki_mk,0.219868,-0.219868
CST_R_dki_fa,0.213258,-0.213258
ILF_L_dki_fa,0.212395,0.212395
UNC_R_dki_fa,0.198812,0.198812
IFO_L_dki_awf,0.184880,-0.184880


                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.076
Model:                            OLS   Adj. R-squared:                  0.035
Method:                 Least Squares   F-statistic:                     1.867
Date:                Thu, 23 Jul 2026   Prob (F-statistic):            0.00412
Time:                        20:06:23   Log-Likelihood:                -82.319
No. Observations:                 691   AIC:                             224.6
Df Residuals:                     661   BIC:                             360.8
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4715      0.029     

In [16]:
from trois_utilities import run_nested_cv_regression
from trois_utilities import run_diffusion_pca


In [17]:
results_5 = run_diffusion_pca(df, n_pcs=25)
results_5["explained_variance"]
results_5["loadings"]["PC8"].abs().sort_values(ascending=False).head(15)
results_5["loadings"]["PC18"].abs().sort_values(ascending=False).head(15)



CGC_L_dki_mk            0.266485
Motor_dki_md            0.231153
PostParietal_dki_md     0.225781
Occipital_dki_md        0.198131
SupParietal_dki_fa      0.196409
Temporal_dki_md         0.192375
Occipital_dki_fa        0.188060
ATR_L_dki_mk            0.182441
CGC_L_dki_awf           0.180739
Motor_dki_mk            0.178309
CGC_R_dki_md            0.176889
PostParietal_dki_fa     0.170520
SupParietal_dki_md      0.169251
PostParietal_dki_awf    0.169250
ILF_L_dki_md            0.163016
Name: PC18, dtype: float64

In [18]:
ridge_results = run_nested_cv_regression(df, run_diffusion_pca(df, n_pcs=25)["pca_df"], model_name="ridge",use_age_average=True)

print(f"Mean nested CV R2: {ridge_results['mean_r2']:.4f} ± {ridge_results['sd_r2']:.4f}")
print(f"Mean nested CV RMSE: {ridge_results['mean_rmse']:.4f} ± {ridge_results['sd_rmse']:.4f}")

ridge_results["feature_weights_summary"].head(20)

Mean nested CV R2: -0.0192 ± 0.0287
Mean nested CV RMSE: 0.2834 ± 0.0173


,feature,mean,std,min,max,bootstrap_mean,bootstrap_ci_low,bootstrap_ci_high
0,PC8,0.014854,0.006091,0.006865,0.022765,0.014915,0.010045,0.019458
1,PC15,0.013424,0.005486,0.006300,0.020284,0.013448,0.009094,0.017451
2,PC18,-0.012034,0.005880,-0.021742,-0.006901,-0.011968,-0.017049,-0.008182
3,PC9,-0.011440,0.002749,-0.016110,-0.009495,-0.011441,-0.013911,-0.009773
4,PC22,0.010416,0.001667,0.008281,0.012295,0.010406,0.009181,0.011711
5,PC14,-0.009734,0.004169,-0.014074,-0.003542,-0.009729,-0.012745,-0.006427
6,PC1,-0.008900,0.005401,-0.014595,-0.002830,-0.008991,-0.013158,-0.004838
7,PC17,-0.008516,0.004580,-0.016259,-0.004196,-0.008541,-0.012594,-0.005530
8,PC19,0.008139,0.002859,0.003716,0.010694,0.008109,0.005618,0.010154
9,PC12,0.007876,0.004194,0.003293,0.014358,0.007881,0.004747,0.011575


In [19]:
# withouth age + sex
ridge_results = run_nested_cv_regression(
    df=df,
    pca_df=pca_df,
    task="regression",
    model_name="ridge",
    use_age_covariate=False,
    use_age_average=False,
    use_sex_covariate=False,
)

print(f"Mean nested CV R2: {ridge_results['mean_r2']:.4f} ± {ridge_results['sd_r2']:.4f}")
print(f"Mean nested CV RMSE: {ridge_results['mean_rmse']:.4f} ± {ridge_results['sd_rmse']:.4f}")

ridge_results["feature_weights_summary"].head(20)

Mean nested CV R2: -0.0175 ± 0.0278
Mean nested CV RMSE: 0.2831 ± 0.0174


,feature,mean,std,min,max,bootstrap_mean,bootstrap_ci_low,bootstrap_ci_high
0,PC8,0.014846,0.006237,0.006849,0.023293,0.014853,0.010313,0.019380
1,PC15,0.013299,0.005445,0.006221,0.020081,0.013269,0.008764,0.017285
2,PC18,-0.011919,0.005921,-0.021736,-0.006871,-0.011936,-0.017009,-0.008090
3,PC9,-0.011591,0.002993,-0.016726,-0.009583,-0.011627,-0.014362,-0.009814
4,PC22,0.010235,0.001668,0.008208,0.012290,0.010218,0.008977,0.011493
5,PC14,-0.009500,0.004072,-0.013501,-0.003406,-0.009504,-0.012435,-0.006306
6,PC1,-0.009109,0.005368,-0.014723,-0.002903,-0.009112,-0.013324,-0.004968
7,PC19,0.008437,0.002847,0.004094,0.010771,0.008412,0.006011,0.010406
8,PC17,-0.008351,0.004468,-0.015846,-0.003974,-0.008349,-0.012327,-0.005494
9,PC5,0.008246,0.004414,0.003534,0.013258,0.008258,0.004872,0.011620


# on test set

## set up test PCAs from train fit

In [28]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd

test_df = pd.read_csv("test_wide_diff_all_data.csv")


# diffusion columns
diffusion_cols = [col for col in df.columns if "dki_" in col]

# train features
X_train_diff = df[diffusion_cols].copy()
X_train_diff = X_train_diff.apply(pd.to_numeric, errors="coerce").fillna(X_train_diff.mean())

# test features
X_test_diff = test_df[diffusion_cols].copy()
X_test_diff = X_test_diff.apply(pd.to_numeric, errors="coerce").fillna(X_train_diff.mean())

# fit scaler on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_diff)

# apply same scaler to test
X_test_scaled = scaler.transform(X_test_diff)

# fit PCA on train only
pca = PCA(n_components=25, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)

# transform test with same PCA fit
X_test_pca = pca.transform(X_test_scaled)

# turn into DataFrames
pca_train = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=train_df.index
)

pca_test = pd.DataFrame(
    X_test_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=test_df.index
)

In [29]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error

train_df = df

# pca_train and pca_test should both come from the SAME PCA fit on train only
# pca_train: PCA scores for training subjects
# pca_test: PCA scores for test subjects

y_train = pd.to_numeric(train_df["DDisc_AUC_40K"], errors="coerce")
y_test = pd.to_numeric(test_df["DDisc_AUC_40K"], errors="coerce")

train_mask = y_train.notna()
test_mask = y_test.notna()

X_train = pca_df.loc[train_mask]
y_train = y_train.loc[train_mask]

X_test = pca_test.loc[test_mask]
y_test = y_test.loc[test_mask]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": np.logspace(-3, 3, 25)
}

inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)

# Coefficient weights for the final fitted ridge model
best_ridge = search.best_estimator_.named_steps["model"]

weight_summary = pd.DataFrame({
    "feature": X_train.columns,
    "weight": best_ridge.coef_
})

weight_summary["abs_weight"] = weight_summary["weight"].abs()
weight_summary = weight_summary.sort_values("abs_weight", ascending=False)

print(weight_summary.head(20))

y_pred = search.predict(X_test)

print("Best alpha:", search.best_params_)
print(f"Test R2: {r2_score(y_test, y_pred):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

   feature    weight  abs_weight
7      PC8  0.011891    0.011891
14    PC15  0.010638    0.010638
8      PC9 -0.010048    0.010048
17    PC18 -0.009735    0.009735
21    PC22  0.008904    0.008904
13    PC14 -0.008079    0.008079
0      PC1 -0.007114    0.007114
16    PC17 -0.007108    0.007108
18    PC19  0.006899    0.006899
4      PC5  0.006238    0.006238
11    PC12  0.006155    0.006155
20    PC21 -0.005777    0.005777
22    PC23 -0.005577    0.005577
10    PC11 -0.004091    0.004091
3      PC4 -0.003699    0.003699
15    PC16 -0.003319    0.003319
6      PC7  0.002858    0.002858
12    PC13 -0.002654    0.002654
24    PC25  0.002228    0.002228
1      PC2 -0.002170    0.002170
Best alpha: {'model__alpha': 1000.0}
Test R2: 0.0077
Test RMSE: 0.2929


# other models

In [30]:
elastic_results = run_nested_cv_regression(df, pca_df, model_name="elastic")
print(f"Mean nested CV R2: {elastic_results['mean_r2']:.4f} ± {elastic_results['sd_r2']:.4f}")
print(f"Mean nested CV RMSE: {elastic_results['mean_rmse']:.4f} ± {elastic_results['sd_rmse']:.4f}")

Mean nested CV R2: -0.0382 ± 0.0378
Mean nested CV RMSE: 0.2859 ± 0.0164


In [ ]:
lasso_results = run_nested_cv_regression(df, pca_df, model_name="lasso")
print(f"Mean nested CV R2: {lasso_results['mean_r2']:.4f} ± {lasso_results['sd_r2']:.4f}")
print(f"Mean nested CV RMSE: {lasso_results['mean_rmse']:.4f} ± {lasso_results['sd_rmse']:.4f}")

In [ ]:
linear_results = run_nested_cv_regression(df, pca_df, model_name="linear")
print(f"Mean nested CV R2: {linear_results['mean_r2']:.4f} ± {linear_results['sd_r2']:.4f}")
print(f"Mean nested CV RMSE: {linear_results['mean_rmse']:.4f} ± {linear_results['sd_rmse']:.4f}")

In [ ]:
rf_results = run_nested_cv_regression(df, pca_df, model_name="rf")
print(f"RF R2: {rf_results['mean_r2']:.4f} ± {rf_results['sd_r2']:.4f}")
print(f"RF RMSE: {rf_results['mean_rmse']:.4f} ± {rf_results['sd_rmse']:.4f}")

In [ ]:
rf_results["feature_weights_summary"].head(20)

In [ ]:
svr_results = run_nested_cv_regression(df, pca_df, model_name="svr")
print(f"RF R2: {svr_results['mean_r2']:.4f} ± {rf_results['sd_r2']:.4f}")
print(f"RF RMSE: {svr_results['mean_rmse']:.4f} ± {rf_results['sd_rmse']:.4f}")

In [ ]:
if svr_results["feature_weights_summary"] is not None:
    print(svr_results["feature_weights_summary"].head(20))
else:
    print("No native feature weights for this model (e.g., SVR with RBF).")

# Predict age

In [ ]:
age_results = run_nested_cv_regression(
    df,
    pca_df,   # or pca_df directly
    task="age_classification",
    model_name="logreg",        # logreg, rf, svc, knn
    include_sex_for_age=False,  # diffusion-only by default
)

print(f"Accuracy: {age_results['mean_accuracy']:.4f} ± {age_results['sd_accuracy']:.4f}")
print(f"Balanced Acc: {age_results['mean_balanced_accuracy']:.4f} ± {age_results['sd_balanced_accuracy']:.4f}")
print(f"F1 macro: {age_results['mean_f1_macro']:.4f} ± {age_results['sd_f1_macro']:.4f}")

age_results["feature_weights_summary"].head(20)